# Week 64, Day 3: Frames and the evaluation loop

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w064d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import sys

def inner():
    frame = sys._getframe()
    names = []
    while frame is not None and frame.f_code.co_name != "<module>":
        names.append(frame.f_code.co_name)
        frame = frame.f_back
    return names

def middle():
    return inner()

def outer():
    return middle()

print(outer())

In [ ]:
import inspect, sys

def show(label):
    frame = sys._getframe()
    caller = frame.f_back
    print(label, "| this:", frame.f_code.co_name, "| locals:", sorted(frame.f_locals),
          "| called from:", caller.f_code.co_name)

def area(w, h):
    result = w * h
    show("in area")
    return result

area(3, 4)

def counter():
    n = 0
    while True:
        n += 1
        yield n

gen = counter()
next(gen)
next(gen)
print(inspect.getgeneratorstate(gen), gen.gi_frame.f_locals["n"])
gen.close()
print(inspect.getgeneratorstate(gen), gen.gi_frame)

In [ ]:
import inspect

def log(message):
    caller = inspect.stack()[1]
    return f"[{caller.function}] {message}"

def checkout(basket):
    return log(f"{len(basket)} items")

print(checkout(["tea", "jam"]))

def depth():
    return len(inspect.stack())

def deeper():
    return depth()

print(deeper() - depth())

In [ ]:
import traceback

def parse(text):
    return int(text)

def load(rows):
    return [parse(r) for r in rows]

def main():
    return load(["1", "two"])

try:
    main()
except ValueError as e:
    tb = e.__traceback__
    chain = []
    while tb is not None:
        chain.append(tb.tb_frame.f_code.co_name)
        tb = tb.tb_next
    print(chain)
    print([summary.name for summary in traceback.extract_tb(e.__traceback__)])
    print(traceback.format_exception_only(e)[-1].strip())

In [ ]:
import sys

def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

def traced(func, *args):
    calls, returns = [], []
    def tracer(frame, event, arg):
        if event == "call" and frame.f_code is func.__code__:
            calls.append(frame.f_locals["n"])
            return tracer                      # follow this frame
        if event == "return":
            returns.append(arg)
        return tracer if event != "call" else None
    previous = sys.gettrace()
    sys.settrace(tracer)
    try:
        result = func(*args)
    finally:
        sys.settrace(previous)
    return result, calls, returns

print(traced(fib, 4))

In [ ]:
def run_frames(functions, entry, args):
    "A tiny machine with call frames: each function is a list of instructions."
    frames = [{"func": entry, "pc": 0, "locals": dict(args), "stack": []}]
    max_depth = 1
    while True:
        frame = frames[-1]
        op, *arg = functions[frame["func"]]["code"][frame["pc"]]
        frame["pc"] += 1
        if op == "LOAD":
            frame["stack"].append(frame["locals"][arg[0]])
        elif op == "PUSH":
            frame["stack"].append(arg[0])
        elif op == "MUL":
            b, a = frame["stack"].pop(), frame["stack"].pop()
            frame["stack"].append(a * b)
        elif op == "CALL":
            name = arg[0]
            params = functions[name]["params"]
            values = [frame["stack"].pop() for _ in params][::-1]
            frames.append({"func": name, "pc": 0, "locals": dict(zip(params, values)), "stack": []})
            max_depth = max(max_depth, len(frames))
        elif op == "RETURN":
            value = frame["stack"].pop()
            frames.pop()
            if not frames:
                return value, max_depth
            frames[-1]["stack"].append(value)

functions = {
    "square": {"params": ["x"], "code": [("LOAD", "x"), ("LOAD", "x"), ("MUL",), ("RETURN",)]},
    "main": {"params": ["n"], "code": [("LOAD", "n"), ("CALL", "square"), ("PUSH", 2), ("MUL",), ("RETURN",)]},
}
print(run_frames(functions, "main", {"n": 5}))

> **Key idea.** A frame holds one call's state: its code object, locals, globals, value stack, current line and a link to its caller (f_back); linked frames form the call stack, and generators keep theirs between steps. sys._getframe and inspect.stack show live frames: use them briefly and don't store them or write to their locals. An exception's __traceback__ is a chain of tb_frame / tb_lineno / tb_next records, outermost first. sys.settrace installs a tracer: the "call" event's return value becomes the frame's local tracer, which gets "line", "return" and "exception" events; always restore sys.gettrace() in a finally. The evaluation loop reads instructions and dispatches on them; Python-to-Python calls push frames without using the C stack, and the loop also handles signals, thread switches and exceptions.

### Exercise: Who called me?

Write `caller_name()`: the name of the function that called the function that called `caller_name` (use `sys._getframe` with the right depth), and `call_chain(stop="<module>")`: the list of function names from the caller of `call_chain` downwards, stopping **before** the first frame whose function is named `stop` (or at the bottom of the stack).

In [ ]:
import sys

def caller_name():
    ___

def call_chain(stop="<module>"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_caller():\n    \"walking frames\"\n    def helper():\n        return caller_name()\n    def report():\n        return helper()\n    assert report() == \"report\"\n    def c():\n        return call_chain(stop=\"test_caller\")\n    def b():\n        return c()\n    def a():\n        return b()\n    assert a() == [\"c\", \"b\", \"a\"]\n    chain = call_chain()\n    assert chain[0] == \"test_caller\"")

### Exercise: A peek at the caller's variables

Write `caller_locals(*names)`: a **copy** of the caller's local variables as a dict, limited to the given names if any are given (missing names are left out). Then `debug(*names)`: a string like `"x=1, total=5"` showing the named variables of the caller, in the order given, with `repr` of each value, and `"?"` for names the caller doesn't have. Delete your frame references before returning.

In [ ]:
import sys

def caller_locals(*names):
    ___

def debug(*names):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_locals():\n    \"reading frame locals\"\n    def work():\n        x, total, label = 1, 5, \"a\"\n        return caller_locals(), caller_locals(\"x\", \"nope\"), debug(\"x\", \"label\", \"missing\")\n    everything, some, text = work()\n    assert everything == {\"x\": 1, \"total\": 5, \"label\": \"a\"} and some == {\"x\": 1}\n    assert text == \"x=1, label='a', missing=?\"\n    def change():\n        y = 1\n        copy = caller_locals()\n        copy[\"y\"] = 99\n        return y\n    assert change() == 1")

### Exercise: Inside a traceback

Write `traceback_chain(exc)`: a list of `(function name, line offset)` pairs for every frame in the exception's traceback, outermost first, where the offset is the traceback's line number minus the function's `co_firstlineno` (so it doesn't depend on where the code sits in a file). Then `raised_in(exc)`: the name of the function where the exception was actually raised (the innermost frame).

In [ ]:
def traceback_chain(exc):
    ___

def raised_in(exc):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tb():\n    \"walking tracebacks\"\n    def parse(text):\n        value = int(text)\n        return value\n    def total(rows):\n        s = 0\n        for r in rows:\n            s += parse(r)\n        return s\n    try:\n        total([\"1\", \"x\"])\n        assert False\n    except ValueError as e:\n        chain = traceback_chain(e)\n        assert [name for name, _ in chain][-2:] == [\"total\", \"parse\"]\n        assert chain[-1][1] == 1 and chain[-2][1] == 3\n        assert raised_in(e) == \"parse\"\n    try:\n        {}[\"k\"]\n    except KeyError as e:\n        assert raised_in(e) == \"test_tb\"")

### Exercise: Which lines ran?

Write `trace_lines(func, *args)`: call `func(*args)` with a tracer installed and return `(result, lines)`, where `lines` lists, in order, the line offsets (the frame's `f_lineno` minus the function's `co_firstlineno`) of every `"line"` event **in that function's own frames** (not in functions it calls). Restore the previous tracer even if `func` raises. Then `coverage(func, *calls)`: run `func` once per argument tuple in `calls` and return the sorted list of distinct line offsets that ran.

In [ ]:
import sys

def trace_lines(func, *args):
    ___

def coverage(func, *calls):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lines():\n    \"sys.settrace\"\n    def grade(score):\n        if score >= 50:\n            return \"pass\"\n        return \"fail\"\n    def helper(n):\n        return n + 1\n    def total(n):\n        t = 0\n        for i in range(n):\n            t += helper(i)\n        return t\n    assert trace_lines(grade, 70) == (\"pass\", [1, 2]) and trace_lines(grade, 10) == (\"fail\", [1, 3])\n    assert trace_lines(total, 2) == (3, [1, 2, 3, 2, 3, 2, 4])\n    assert coverage(grade, (70,)) == [1, 2] and coverage(grade, (70,), (5,)) == [1, 2, 3]\n    before = sys.gettrace()\n    def boom():\n        raise ValueError\n    try:\n        trace_lines(boom)\n        assert False\n    except ValueError:\n        pass\n    assert sys.gettrace() is before")

### Exercise: Counting calls and depth

Write `call_stats(func, *args)`: run `func(*args)` with a tracer and return `(result, stats)`, where `stats` is a dict from function name to the number of calls made to functions with that name **while `func` ran** (including `func` itself), counting only Python functions whose code object's `co_filename` is the same as `func`'s, plus the key `"max_depth"`: the deepest nesting of those calls (1 for `func` alone). Use `"call"` and `"return"` events to track the depth, and restore the previous tracer.

In [ ]:
import sys

def call_stats(func, *args):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_calls():\n    \"tracing calls\"\n    def fact(n):\n        return 1 if n <= 1 else n * fact(n - 1)\n    def twice(x):\n        return fact(x) + fact(x - 1)\n    result, stats = call_stats(fact, 5)\n    assert result == 120 and stats == {\"fact\": 5, \"max_depth\": 5}\n    result, stats = call_stats(twice, 3)\n    assert result == 8 and stats == {\"twice\": 1, \"fact\": 5, \"max_depth\": 4}\n    result, stats = call_stats(lambda: len([1, 2]))\n    assert result == 2 and stats == {\"<lambda>\": 1, \"max_depth\": 1}")

### Bug hunt: One frame too close

`whoami` wants the name of the function that called it, but `sys._getframe(0)` is `whoami`'s own frame, so this prints `whoami`. Look one frame up so it prints `checkout`.

In [ ]:
import sys

def whoami():
    return sys._getframe(0).f_code.co_name

def checkout():
    return whoami()

print(checkout())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the caller's frame\"\n    expect_output(\"checkout\")")

### Bug hunt: A tracer that stops listening

The tracer handles the `"call"` event but returns `None`, which tells Python not to trace inside that frame, so no `"line"` events arrive and this prints `0`. Return a local tracer that counts lines (and returns itself), so this prints `3`.

In [ ]:
import sys

count = 0

def tracer(frame, event, arg):
    global count
    if event == "line":
        count += 1
    return None

def work():
    a = 1
    b = 2
    return a + b

sys.settrace(tracer)
work()
sys.settrace(None)
print(count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"return the tracer\"\n    expect_output(\"3\")")

### Bug hunt: A tracer left running

The tracer is never removed after the first call, so it keeps counting lines in the second call too, and this prints `[2, 4]`. Remove it after the measured call (restoring the previous tracer) so the second count stays the same, and this prints `[2, 2]`.

In [ ]:
import sys

lines = 0

def tracer(frame, event, arg):
    global lines
    if event == "line":
        lines += 1
    return tracer

def step():
    x = 1
    return x

previous = sys.gettrace()
sys.settrace(tracer)
step()
counts = [lines]
step()
counts.append(lines)
sys.settrace(previous)
print(counts)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"restore the tracer\"\n    expect_output(\"[2, 2]\")")

### Puzzle: Tracing safely

Arrange the lines of a helper that runs a function under a tracer and always restores the previous one. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the frame attribute linking to the caller's frame
q = ___   # True or False: every call to the same function shares one frame
r = ___   # which comes first in exc.__traceback__: "outermost" or "innermost"
s = ___   # what a "call" tracer should return to ignore a frame
t = ___   # True or False: a finished generator still has its frame (gi_frame)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"f_back\" and q is False and r == \"outermost\", \"p, q and r\"\n    assert s is None and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a machine with real frames

Build a stack machine with functions and call frames. A program is a dict from function name to `{"params": [...], "code": [...]}`. Write `call(program, name, *args, max_depth=100)` that runs function `name` and returns its result. Each **frame** has its own locals, value stack and program counter. Instructions:

- `("PUSH", v)`, `("LOAD", name)`, `("STORE", name)`, `("ADD",)`, `("SUB",)`, `("MUL",)`, `("LT",)`, `("EQ",)`;
- `("JUMP", target)` and `("JUMP_IF_FALSE", target)` within the current function's code;
- `("CALL", fname)`: pop as many values as `fname` has parameters (the last pushed is the last argument), push a new frame, and continue there;
- `("RETURN",)`: pop the return value, pop the frame and push the value onto the caller's stack (or finish, if it was the first frame).

Errors: calling an unknown function raises `NameError`; having more than `max_depth` frames at once raises `RecursionError`; reading an unset local raises `NameError`. Write the machine as `call_with_stats(program, name, *args, max_depth=100)`, which returns `(result, calls, deepest)`: the total number of frames created and the maximum depth reached. Do it **without** Python recursion, so deep (but allowed) recursion in the machine can't hit Python's own limit.

In [ ]:
def call_with_stats(program, name, *args, max_depth=100):
    ___

def call(program, name, *args, max_depth=100):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_frames():\n    \"frames and calls\"\n    fact = {\"fact\": {\"params\": [\"n\"], \"code\": [\n        (\"LOAD\", \"n\"), (\"PUSH\", 2), (\"LT\",), (\"JUMP_IF_FALSE\", 6),\n        (\"PUSH\", 1), (\"RETURN\",),\n        (\"LOAD\", \"n\"), (\"LOAD\", \"n\"), (\"PUSH\", 1), (\"SUB\",), (\"CALL\", \"fact\"), (\"MUL\",), (\"RETURN\",)]}}\n    assert call(fact, \"fact\", 5) == 120 and call_with_stats(fact, \"fact\", 5) == (120, 5, 5)\n    assert call(fact, \"fact\", 900, max_depth=1000) > 0\n    try:\n        call(fact, \"fact\", 50, max_depth=10)\n        assert False\n    except RecursionError:\n        pass\n    fib = {\"fib\": {\"params\": [\"n\"], \"code\": [\n        (\"LOAD\", \"n\"), (\"PUSH\", 2), (\"LT\",), (\"JUMP_IF_FALSE\", 6), (\"LOAD\", \"n\"), (\"RETURN\",),\n        (\"LOAD\", \"n\"), (\"PUSH\", 1), (\"SUB\",), (\"CALL\", \"fib\"),\n        (\"LOAD\", \"n\"), (\"PUSH\", 2), (\"SUB\",), (\"CALL\", \"fib\"), (\"ADD\",), (\"RETURN\",)]}}\n    assert call_with_stats(fib, \"fib\", 6) == (8, 25, 6)\n\ndef test_args_and_errors():\n    \"arguments and errors\"\n    prog = {\n        \"minus\": {\"params\": [\"a\", \"b\"], \"code\": [(\"LOAD\", \"a\"), (\"LOAD\", \"b\"), (\"SUB\",), (\"RETURN\",)]},\n        \"main\": {\"params\": [], \"code\": [(\"PUSH\", 10), (\"PUSH\", 3), (\"CALL\", \"minus\"), (\"STORE\", \"r\"),\n                                         (\"LOAD\", \"r\"), (\"LOAD\", \"r\"), (\"MUL\",), (\"RETURN\",)]},\n        \"bad\": {\"params\": [], \"code\": [(\"CALL\", \"nothing\"), (\"RETURN\",)]},\n        \"unset\": {\"params\": [], \"code\": [(\"LOAD\", \"x\"), (\"RETURN\",)]},\n    }\n    assert call(prog, \"main\") == 49 and call(prog, \"minus\", 1, 5) == -4\n    for name in (\"bad\", \"unset\", \"missing\"):\n        try:\n            call(prog, name)\n            assert False\n        except NameError:\n            pass")

## Warm-up

Each call made a frame, and each frame points back to its caller's frame with `f_back`. Following those links from the innermost call gives the call stack, the same list a traceback shows you after an error.

## What a frame holds

When a function is called, CPython creates a **frame** for that call. It holds:

- `f_code`: the code object being run (the same one for every call to that function);
- `f_locals`: the call's local variables (parameters included);
- `f_globals` and `f_builtins`: the namespaces used for global and built-in names;
- `f_back`: the frame of the **caller**;
- `f_lineno`: the line currently running, and `f_lasti`, the offset of the current instruction;
- internally, the **value stack** that bytecode instructions push to and pop from (yesterday), and the stack of active `try` blocks.

Frames are linked into the **call stack**: the newest frame at the top, `f_back` leading down to the module-level frame. When a function returns, its frame is popped and (unless something still refers to it) freed. A recursive function has one frame per active call, which is why deep recursion costs memory and why Python limits it (`sys.getrecursionlimit()`, normally 1000).

Generators are the exception: a generator **keeps its frame** between `next()` calls (`gen.gi_frame`), which is exactly how it remembers its local variables and where it paused (Week 59).

## Looking at the stack

`sys._getframe(depth=0)` returns the current frame, or the one `depth` levels up (1 is the caller). The leading underscore marks it as CPython-specific; the portable equivalents are in `inspect`:

- `inspect.currentframe()` (may return `None` on other Pythons);
- `inspect.stack()`: a list of `FrameInfo` records for the whole stack, newest first, each with `function`, `lineno`, `filename` and `frame`;
- `inspect.getframeinfo(frame)`.

Uses: logging that records the calling function, debugging helpers, and frameworks that look up the caller's variables. Two cautions:

- **Don't keep frames around.** A frame refers to all its locals, and the locals may refer back to the frame, making a cycle (Week 62). Use what you need and drop the reference (or `del frame`).
- **Don't change locals through a frame.** Since Python 3.13, `frame.f_locals` for a function is a live view that does write through to the variables (PEP 667), but older versions behaved differently, and code that relies on either is fragile.

## Tracebacks are frames

When an exception escapes a function, Python builds a **traceback**: a linked list of traceback objects, one for each frame the exception passed through. `exc.__traceback__` is the first (the outermost frame); each has:

- `tb_frame`: the frame;
- `tb_lineno`: the line in that frame where the exception passed;
- `tb_next`: the next traceback object, one call deeper, or `None` at the frame where the exception was raised.

The `traceback` module formats these (`traceback.format_exception(exc)`, `traceback.extract_tb(tb)`), and `traceback.walk_tb(tb)` yields `(frame, lineno)` pairs. Because a traceback holds frames, and frames hold all their locals, keeping an exception object alive keeps all of those alive too (Week 62).

## Tracing with sys.settrace

`sys.settrace(tracer)` installs a **trace function** that the evaluation loop calls as code runs. It's how `pdb` (the debugger), coverage tools and some profilers work.

- The **global** tracer is called with `(frame, "call", None)` whenever a new frame starts. What it **returns** becomes the **local** tracer for that frame (return `None` to ignore the frame).
- The local tracer is then called with events for that frame: `"line"` (a new line is about to run), `"return"` (with the return value as `arg`), and `"exception"` (with the exception details). It must return itself (or another function) to keep receiving events.
- Tracing only affects the current thread, slows everything down a lot, and must be **removed** afterwards: save `sys.gettrace()` first and restore it in a `finally`, so you don't break a debugger or coverage tool that was already running.

(Python 3.12 added `sys.monitoring`, a faster, lower-level API for tools; `settrace` remains the simple, portable choice.)

## The evaluation loop

At the heart of CPython is one large C function, `_PyEval_EvalFrameDefault` in `Python/ceval.c`. Stripped to its essentials, it's yesterday's stack machine:

```
for (;;) {
    instruction = *next_instr++;
    switch (opcode(instruction)) {
        case LOAD_FAST:   push(frame->locals[oparg]); break;
        case BINARY_OP:   right = pop(); left = pop(); push(binary_op(left, right, oparg)); break;
        case CALL:        ... push a new frame and continue the loop in it ...
        case RETURN_VALUE: ... pop the frame, push the result onto the caller's stack ...
        ...
    }
}
```

Some details matter for understanding Python's behaviour:

- **Calls between Python functions don't use the C stack** (since 3.11): a call pushes a new frame and the same loop continues with it, which made calls faster and deep recursion cheaper.
- **The loop checks for interruptions** regularly: signals such as Ctrl+C (raising `KeyboardInterrupt`), and, in builds with the global interpreter lock, requests from other threads to take a turn (Week 66).
- **Exceptions** make the loop look up the current instruction in the code object's **exception table** to find the handler; if there is none, the frame is popped and the exception continues in the caller, adding a traceback entry.
- In Python 3.13 and later, an experimental **JIT compiler** can turn hot sequences of instructions into machine code, but the loop remains the reference behaviour.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, put `breakpoint()` inside a function, run it, and type `where` (or `w`) at the `(Pdb)` prompt to see the stack of frames, then `up` and `down` to move between them. Tomorrow: what objects look like inside CPython, in C.